# Pangaea Benchmark — Results Analysis
Load results from local experiment directories and build pivot tables.

In [ ]:
import re
import os
from pathlib import Path
import pandas as pd
import numpy as np

EXPERIMENTS_BASE = Path("/dss/dsstbyfs02/pn49cu/pn49cu-dss-0001/niklas/experiments")

# ── Display name mappings ────────────────────────────────────────────────────
ENCODER_LABELS = {
    "dofa":               "DOFA",
    "dofav2":             "DOFAv2",
    "carl":               "CARL",
    "panopticon":         "Panopticon",
    "copernicus_fm_base": "Copernicus-FM",
    "smarties_vitb":      "SMARTIES",
    "dinov2_vitb14":      "DINOv2",
    "unet_encoder":       "UNet",
    "unet_encoder_mi":    "UNet",
    "vit_scratch":        "ViT-Scratch",
    "vit_scratch_mi":     "ViT-Scratch",
}

DATASET_LABELS = {
    "hlsburnscars":      "HLS-BurnScars",
    "sen1floods11":      "Sen1Floods11",
    "mados":             "MADOS",
    "pastis":            "PASTIS",
    "dynamicen":         "DynamicEarthNet",
    "croptypemapping":   "CropTypeMapping",
    "spacenet7cd":       "SpaceNet7-CD",
    "ai4smallfarms":     "AI4SmallFarms",
    "fivebillionpixels": "FiveBillionPixels",
}

DATASET_ORDER = list(DATASET_LABELS.keys())

# Encoders listed in longest-first order for greedy name matching
KNOWN_ENCODERS = [
    "copernicus_fm_base", "copernicus_fm_large",
    "smarties_vitb", "dinov2_vitb14",
    "dofav2", "dofa", "panopticon", "carl",
    "vit_scratch_mi", "vit_scratch",
    "unet_encoder_mi", "unet_encoder",
]

KNOWN_DECODERS = [
    "seg_upernet_mt_ltae", "seg_upernet",
    "seg_siamupernet_conc", "seg_siamupernet_diff",
    "seg_siam_mlp_fusion", "seg_mlp_fusion",
    "seg_siamunet_conc", "seg_siamunet_diff", "seg_unet",
    "seg_linear_probe", "seg_siam_linear_probe",
    "cls_linear_probe",
]

DATASET_RE = re.compile(
    r"_(" + "|".join(re.escape(d) for d in DATASET_ORDER) + r")_"
)

print("Config loaded.")

In [ ]:
# ── Core parsing functions ───────────────────────────────────────────────────

def parse_run_dir_name(name: str) -> dict:
    """Extract dataset, encoder, decoder from a run directory name."""
    m = DATASET_RE.search(name)
    if not m:
        return {}
    dataset = m.group(1)
    rest = name[m.end():]

    encoder = next((e for e in KNOWN_ENCODERS if rest.startswith(e + "_")), None)
    if encoder:
        rest = rest[len(encoder) + 1:]

    decoder = next((d for d in KNOWN_DECODERS if rest.startswith(d)), None)

    return {"dataset": dataset, "encoder": encoder or "", "decoder": decoder or ""}


def parse_log_metrics(log_path: Path) -> dict:
    """
    Parse train.log-0 for test metrics.

    Extracts the Mean values from [test] IoU, F1, Precision, Recall sections.
    Returns a dict with keys: test_mIoU, test_mF1, test_mPrecision, test_mRecall.
    For change detection (IoU section contains exactly 2 classes), also returns test_IoU.
    """
    if not log_path.exists():
        return {}

    text = log_path.read_text(errors="ignore")

    # Find all [test] sections — there may be multiple if val was run at test time
    # We want the LAST occurrence (final test run)
    sections = re.split(r"\[test\] ------- IoU", text)
    if len(sections) < 2:
        return {}
    test_block = "[test] ------- IoU" + sections[-1]

    metrics = {}
    metric_map = {
        "IoU":       "test_mIoU",
        "F1-score":  "test_mF1",
        "Precision": "test_mPrecision",
        "Recall":    "test_mRecall",
    }

    for section_name, metric_key in metric_map.items():
        # Find the Mean line inside this section
        pattern = rf"\[test\] ------- {section_name}.*?\[test\] Mean\s+([\d.]+)"
        m = re.search(pattern, test_block, re.DOTALL)
        if m:
            metrics[metric_key] = float(m.group(1))

    return metrics


def load_experiment_folder(
    folder: str | Path,
    encoder_map: dict | None = None,
    dataset_map: dict | None = None,
    metric: str = "test_mIoU",
    keep_all_metrics: bool = False,
) -> pd.DataFrame:
    """
    Load all runs from an experiment folder and return a long-form DataFrame.

    Parameters
    ----------
    folder : path to an experiment directory (e.g. pangaea_full_seg/upernet/panopticon_seg)
             OR a parent directory containing multiple encoder subdirectories.
             The function auto-detects whether run dirs are direct children or
             one level deeper.
    encoder_map : optional override for encoder display names {raw: display}
    dataset_map : optional override for dataset display names {raw: display}
    metric : which metric column to extract (default: test_mIoU)
    keep_all_metrics : if True, keep all parsed metrics instead of just `metric`

    Returns
    -------
    Long-form DataFrame with columns: run_dir, encoder, decoder, dataset, [metrics...]
    """
    enc_map = {**ENCODER_LABELS, **(encoder_map or {})}
    ds_map  = {**DATASET_LABELS,  **(dataset_map  or {})}
    folder  = Path(folder)

    # Collect candidate run directories:
    # A run dir contains train.log-0. Walk up to 2 levels deep.
    run_dirs = []
    for child in sorted(folder.iterdir()):
        if not child.is_dir():
            continue
        if (child / "train.log-0").exists():
            run_dirs.append(child)
        else:
            for grandchild in sorted(child.iterdir()):
                if grandchild.is_dir() and (grandchild / "train.log-0").exists():
                    run_dirs.append(grandchild)

    rows = []
    for run_dir in run_dirs:
        parsed = parse_run_dir_name(run_dir.name)
        if not parsed.get("dataset"):
            continue
        log_metrics = parse_log_metrics(run_dir / "train.log-0")
        if not log_metrics:
            continue

        enc_raw = parsed["encoder"]
        ds_raw  = parsed["dataset"]

        row = {
            "run_dir": run_dir.name,
            "encoder_raw": enc_raw,
            "decoder":     parsed["decoder"],
            "dataset_raw": ds_raw,
            "encoder":     enc_map.get(enc_raw, enc_raw),
            "dataset":     ds_map.get(ds_raw, ds_raw),
        }
        if keep_all_metrics:
            row.update(log_metrics)
        elif metric in log_metrics:
            row[metric] = log_metrics[metric]
        else:
            continue
        rows.append(row)

    return pd.DataFrame(rows)


def to_pivot(
    df: pd.DataFrame,
    metric: str = "test_mIoU",
    index: str = "encoder",
    columns: str = "dataset",
    aggfunc: str = "max",
    dataset_order: list | None = None,
    encoder_order: list | None = None,
    add_mean: bool = True,
) -> pd.DataFrame:
    """
    Pivot a long-form results DataFrame to encoder × dataset.

    Parameters
    ----------
    df           : output of load_experiment_folder()
    metric       : value column to pivot
    index        : row grouping (default: encoder display name)
    columns      : column grouping (default: dataset display name)
    aggfunc      : how to aggregate duplicates — 'max', 'mean', or 'last'
    dataset_order: explicit column order; defaults to DATASET_LABELS order
    encoder_order: explicit row order
    add_mean     : whether to add a Mean column (over available values)
    """
    if metric not in df.columns:
        raise ValueError(f"Metric '{metric}' not in DataFrame columns: {list(df.columns)}")

    pivot = df.pivot_table(
        index=index, columns=columns, values=metric,
        aggfunc=aggfunc
    )

    # Apply column order
    if dataset_order is None:
        ordered_ds = [DATASET_LABELS[d] for d in DATASET_ORDER if DATASET_LABELS[d] in pivot.columns]
        remaining  = [c for c in pivot.columns if c not in ordered_ds]
        dataset_order = ordered_ds + remaining
    pivot = pivot.reindex(columns=[c for c in dataset_order if c in pivot.columns])

    # Apply row order
    if encoder_order is not None:
        pivot = pivot.reindex([e for e in encoder_order if e in pivot.index])

    if add_mean:
        pivot["Mean"] = pivot.mean(axis=1, skipna=True).round(3)

    return pivot.round(2)


print("Functions defined: load_experiment_folder(), to_pivot()")

## Full-band UPerNet results

In [ ]:
df_upernet = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_full_seg" / "upernet",
    metric="test_mIoU",
)
print(f"{len(df_upernet)} runs loaded")
df_upernet.groupby(["encoder", "dataset"])["test_mIoU"].max().unstack()

In [ ]:
pivot_upernet = to_pivot(df_upernet)
pivot_upernet

## Full-band MLP Fusion results

In [ ]:
df_mlp = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_full_seg" / "mlp_fusion",
    metric="test_mIoU",
)
print(f"{len(df_mlp)} runs loaded")
pivot_mlp = to_pivot(df_mlp)
pivot_mlp

## RGB UPerNet results

In [ ]:
df_rgb_up = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_rgb" / "upernet",
    metric="test_mIoU",
)
print(f"{len(df_rgb_up)} runs loaded")
pivot_rgb_up = to_pivot(df_rgb_up)
pivot_rgb_up

## RGB MLP Fusion results

In [ ]:
df_rgb_mlp = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_rgb" / "mlp_fusion",
    metric="test_mIoU",
)
print(f"{len(df_rgb_mlp)} runs loaded")
pivot_rgb_mlp = to_pivot(df_rgb_mlp)
pivot_rgb_mlp

## Advanced usage examples

In [ ]:
# Load a single encoder folder
df_panopticon = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_full_seg" / "upernet" / "panopticon_seg",
    keep_all_metrics=True,  # get mIoU, mF1, Precision, Recall
)
df_panopticon[["dataset", "test_mIoU", "test_mF1", "test_mPrecision", "test_mRecall"]]

In [ ]:
# Compare UPerNet vs MLP fusion side-by-side for full-band runs
mean_up  = to_pivot(df_upernet)[["Mean"]].rename(columns={"Mean": "UPerNet"})
mean_mlp = to_pivot(df_mlp)[["Mean"]].rename(columns={"Mean": "MLP Fusion"})
comparison = mean_up.join(mean_mlp, how="outer")
comparison["Δ (UPerNet - MLP)"] = (comparison["UPerNet"] - comparison["MLP Fusion"]).round(2)
comparison.sort_values("UPerNet", ascending=False)

In [ ]:
# Check which runs are still missing (NaN cells)
print("=== Missing UPerNet runs ===")
missing = pivot_upernet.drop(columns="Mean").isna().stack()
print(missing[missing].reset_index()[["encoder", "dataset"]].to_string(index=False))

print("\n=== Missing MLP Fusion runs ===")
missing_mlp = pivot_mlp.drop(columns="Mean").isna().stack()
print(missing_mlp[missing_mlp].reset_index()[["encoder", "dataset"]].to_string(index=False))